### 1. Generazione Sintetica e Bilanciamento del Dataset (Albumentations)
Questo blocco di codice si occupa di risolvere lo sbilanciamento tra immagini normali e fisheye creando nuove immagini distorte artificialmente, senza dover raccogliere nuovi dati reali. Ecco i passaggi chiave eseguiti dallo script:

* **Calcolo del Bilanciamento (50/50):** il codice conta quante immagini `"Normal"` e `"Fisheye"` reali ci sono nella cartella di addestramento (dello split2). Calcola poi la differenza esatta per capire quante immagini sintetiche generare, in modo da avere un rapporto finale perfettamente bilanciato.
* **Campionamento Casuale:** seleziona in modo randomico le immagini a prospettiva lineare su cui applicare la deformazione.
* **Sanitizzazione dei Dati (Clipping):** prima di applicare la trasformazione, legge i file `.txt` di YOLO. Se a causa di vecchie annotazioni imperfette un bounding box sfora le dimensioni dell'immagine (es. un valore `x = 1.00039`), lo script "taglia" matematicamente l'eccesso costringendolo entro il limite massimo consentito (`1.0`), prevenendo errori fatali della libreria.
* **Distorsione Ottica (Albumentations):** utilizza la funzione `"OpticalDistortion"` per incurvare i pixel simulando l'effetto a barile di una lente grandangolare. Contemporaneamente, ricalcola e sposta le coordinate dei bounding box, eliminando quelli che finiscono troppo fuori dall'inquadratura (`min_visibility = 0.2`).

In [ ]:
import os
import cv2
import random
import albumentations as A

print("=== GENERAZIONE SINTETICA BILANCIATA ===\n")

# --- 1. CONFIGURAZIONE PERCORSI ---
# !!! ATTENZIONE !!!
# modificare i path
cartella_immagini_input = "/inserisci/il/path/per/pp4av-data-yolo-split2/pp4av_yolo_format/images/train"
cartella_labels_input = "/inserisci/il/path/per/pp4av-data-yolo-split2/pp4av_yolo_format/labels/train"

cartella_immagini_output = "/inserisci/il/path/per/synthetic_fisheye_albu/images/train"
cartella_labels_output = "/inserisci/il/path/per/synthetic_fisheye_albu/labels/train"

os.makedirs(cartella_immagini_output, exist_ok = True)
os.makedirs(cartella_labels_output, exist_ok = True)

# --- 2. ANALISI DEL DATASET E CALCOLO BILANCIAMENTO ---
immagini_totali = os.listdir(cartella_immagini_input)

immagini_normal = [img for img in immagini_totali if img[0].isalpha() and (img.endswith(".png") or img.endswith(".jpg"))]
immagini_fisheye_reali = [img for img in immagini_totali if not img[0].isalpha() and (img.endswith(".png") or img.endswith(".jpg"))]

num_normal = len(immagini_normal)
num_fisheye_reali = len(immagini_fisheye_reali)
num_da_generare = num_normal - num_fisheye_reali

print(f"Target per bilanciamento 50/50: {num_da_generare} immagini sintetiche da generare.\n")

random.seed(42)
immagini_da_distorcere = random.sample(immagini_normal, num_da_generare)

# --- 3. MOTORE MATEMATICO (ALBUMENTATIONS) ---
# A.Compose crea una pipeline che unisce la manipolazione dei pixel al ricalcolo delle etichette
trasformazione = A.Compose(
    
    # 1. DEFORMAZIONE VISIVA (PIXEL)
    # OpticalDistortion simula la distorsione sferica (a barile) tipica delle lenti grandangolari.
    # - distort_limit = 0.8: applica una curvatura radiale molto severa, spingendo i pixel verso l'esterno.
    # - p = 1.0: imposta la probabilità di esecuzione al 100% (ogni immagine in input verrà distorta).
    [A.OpticalDistortion(distort_limit = 0.8, p = 1.0)],
    
    # 2. ADATTAMENTO GEOMETRICO (COORDINATE)
    # BboxParams istruisce l'algoritmo su come spostare i rettangoli per farli coincidere con gli oggetti distorti.
    # - format = "yolo": specifica che i dati in ingresso usano la normalizzazione standard (x_center, y_center, w, h).
    # - min_visibility = 0.2: filtro di sicurezza (Clipping dinamico). Se lo stiramento dell'immagine spinge un 
    #   veicolo o un volto fuori dall'inquadratura lasciandone visibile meno del 20%, il bounding box viene 
    #   automaticamente eliminato. Questo previene l'inserimento di "rumore" o falsi positivi nel Training Set.
    # - label_fields = ["class_labels"]: assicura che l'ID numerico della classe (es. 0 per i volti, 1 per le targhe) 
    #   rimanga rigidamente ancorato al suo rettangolo corrispondente dopo il ricalcolo spaziale.
    bbox_params = A.BboxParams(format = "yolo", min_visibility = 0.2, label_fields = ["class_labels"])
)

# --- 4. CICLO DI ELABORAZIONE ---
contatore_generate = 0

for nome_img in immagini_da_distorcere:
    percorso_img = os.path.join(cartella_immagini_input, nome_img)
    percorso_txt = os.path.join(cartella_labels_input, nome_img.replace(".png", ".txt").replace(".jpg", ".txt"))
    
    if not os.path.exists(percorso_txt):
        continue

    immagine = cv2.imread(percorso_img)
    immagine = cv2.cvtColor(immagine, cv2.COLOR_BGR2RGB)
    
    bboxes = []
    class_labels = []
    
    with open(percorso_txt, "r") as file_label:
        righe = file_label.readlines()
        for riga in righe:
            dati = riga.strip().split()
            classe = int(dati[0])
            x_c = float(dati[1])
            y_c = float(dati[2])
            w = float(dati[3])
            h = float(dati[4])
            
            # --- FASE DI CLIPPING ---
            # Calcoliamo i vertici del rettangolo
            x_min = x_c - (w / 2.0)
            x_max = x_c + (w / 2.0)
            y_min = y_c - (h / 2.0)
            y_max = y_c + (h / 2.0)
            
            # Tagliamo gli eccessi se sforano il range [0.0, 1.0]
            x_min = max(0.0, min(1.0, x_min))
            x_max = max(0.0, min(1.0, x_max))
            y_min = max(0.0, min(1.0, y_min))
            y_max = max(0.0, min(1.0, y_max))
            
            # Riconvertiamo nel formato YOLO
            w_nuovo = x_max - x_min
            h_nuovo = y_max - y_min
            xc_nuovo = x_min + (w_nuovo / 2.0)
            yc_nuovo = y_min + (h_nuovo / 2.0)
            
            # Teniamo il box solo se ha un'area valida dopo il taglio
            if w_nuovo > 0.001 and h_nuovo > 0.001:
                bboxes.append([xc_nuovo, yc_nuovo, w_nuovo, h_nuovo])
                class_labels.append(classe)
                
    if len(bboxes) > 0:
        risultato = trasformazione(image = immagine, bboxes = bboxes, class_labels = class_labels)
        immagine_distorta = risultato["image"]
        bboxes_distorti = risultato["bboxes"]
        classi_distorte = risultato["class_labels"]
        
        nome_img_synth = "synth_albu_" + nome_img
        percorso_out_img = os.path.join(cartella_immagini_output, nome_img_synth)
        cv2.imwrite(percorso_out_img, cv2.cvtColor(immagine_distorta, cv2.COLOR_RGB2BGR))
        
        nome_txt_synth = "synth_albu_" + nome_img.replace(".png", ".txt").replace(".jpg", ".txt")
        percorso_out_txt = os.path.join(cartella_labels_output, nome_txt_synth)
        
        with open(percorso_out_txt, "w") as f_out:
            for i in range(len(bboxes_distorti)):
                box = bboxes_distorti[i]
                cls = classi_distorte[i]
                riga_out = f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n"
                f_out.write(riga_out)
                
        contatore_generate += 1
        if contatore_generate % 50 == 0:
            print(f"Progresso: {contatore_generate} / {num_da_generare} immagini elaborate...")

print(f"\nOperazione completata con successo! Generate {contatore_generate} immagini sintetiche.")

### 2. Fusione del nuovo Dataset e creazione dei file YAML (Split 3)
Questo blocco di codice crea l'ambiente di addestramento definitivo per l'approccio Data-Centric, assemblando i dati in modo da prevenire la contaminazione (Data Leakage) e istruendo YOLOv9 su dove trovare i file.

* **Creazione dell'Infrastruttura:** genera una nuova directory pulita denominata `"dataset_split3"` contenente tutte le sottocartelle standard di YOLO (`"train"`, `"val"`, `"test_normal"`, `"test_fisheye"`).
* **Prevenzione del Data Leakage:** copia tutti i dati dallo `"split2"` originale. La regola fondamentale applicata qui è che i set di validazione e di test non vengono minimamente alterati, garantendo che l'esame finale del modello avvenga esclusivamente su immagini reali e intatte.
* **Iniezione del Dominio Sintetico:** i dati distorti artificialmente, creati nel blocco precedente, vengono riversati **solo ed esclusivamente** all'interno della cartella `"train"`. In questo modo, il Training Set diventa massiccio e geometricamente bilanciato.
* **Generazione Dinamica degli YAML:** utilizza la libreria Python `yaml` per creare automaticamente i tre file di configurazione necessari al training e ai test (`"dataset_train_split3.yaml"`, `"dataset_test_normal_split3.yaml"`, `"dataset_test_fisheye_split3.yaml"`).

In [ ]:
import os
import shutil
import yaml

print("=== CREAZIONE DEL NUOVO DATASET E YAML (CICLO 3) ===")

# --- 1. CONFIGURAZIONE PERCORSI ---
cartella_originale = "/inserisci/il/path/per/pp4av-data-yolo-split2/pp4av_yolo_format"
cartella_sintetica = "/inserisci/il/path/per/synthetic_fisheye_albu"
cartella_split3 = "/inserisci/il/path/per/dataset_split3"

# --- 2. CREAZIONE DELLA STRUTTURA DEL NUOVO DATASET ---
print("Creazione cartelle...")
cartelle_da_creare = [
    "images/train", "images/val", "images/test_normal", "images/test_fisheye", 
    "labels/train", "labels/val", "labels/test_normal", "labels/test_fisheye"
]

for cartella in cartelle_da_creare:
    os.makedirs(os.path.join(cartella_split3, cartella), exist_ok = True)

# --- 3. COPIA DEI DATI ORIGINALI ---
print("Copia dei dati originali di addestramento, validazione e test...")
for tipo in ["images", "labels"]:
    for split in ["train", "val", "test_normal", "test_fisheye"]:
        src_dir = os.path.join(cartella_originale, tipo, split)
        dst_dir = os.path.join(cartella_split3, tipo, split)
        
        # Copiamo solo se la cartella originale esiste
        if os.path.exists(src_dir):
            file_list = os.listdir(src_dir)
            for file_name in file_list:
                shutil.copy2(os.path.join(src_dir, file_name), os.path.join(dst_dir, file_name))

# --- 4. INIEZIONE DEI DATI SINTETICI (IL BILANCIAMENTO) ---
print("Iniezione delle immagini e labels sintetiche nel Training Set...")
for tipo in ["images", "labels"]:
    src_dir = os.path.join(cartella_sintetica, tipo, "train")
    dst_dir = os.path.join(cartella_split3, tipo, "train")
    
    if os.path.exists(src_dir):
        file_list = os.listdir(src_dir)
        for file_name in file_list:
            shutil.copy2(os.path.join(src_dir, file_name), os.path.join(dst_dir, file_name))

# --- 5. CREAZIONE DEI 3 FILE YAML ---
print("Generazione dei file YAML per il Ciclo 3...")

# Yaml per il Training
yaml_train = {
    "path": cartella_split3, 
    "train": "images/train", 
    "val": "images/val",
    "nc": 2, 
    "names": ["face", "license_plate"]
}

# Yaml per Test Immagini Normali
yaml_test_normal = {
    "path": cartella_split3, 
    "train": "images/train", 
    "val": "images/test_normal",
    "nc": 2, 
    "names": ["face", "license_plate"]
}

# Yaml per Test Immagini Fisheye
yaml_test_fisheye = {
    "path": cartella_split3, 
    "train": "images/train", 
    "val": "images/test_fisheye",
    "nc": 2, 
    "names": ["face", "license_plate"]
}

# Salvataggio dei file
# !!! ATTENZIONE !!!
# modificare i path
with open("/inserisci/il/path/di/output/per/dataset_train_split3.yaml", 'w') as f: yaml.dump(yaml_train, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_normal_split3.yaml", 'w') as f: yaml.dump(yaml_test_normal, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_fisheye_split3.yaml", 'w') as f: yaml.dump(yaml_test_fisheye, f, sort_keys = False)

print("\nOperazione completata con successo! Il nuovo Dataset è pronto.")
print("File YAML generati: \"dataset_train_split3.yaml\", \"dataset_test_normal_split3.yaml\", \"dataset_test_fisheye_split3.yaml\"")